In [0]:
# Parâmetros via Databricks Widgets (1 notebook, 3 jobs com params diferentes)
# Para Teste
#precisa configurar IPCA, SELIC e CDI no WORKFLOW
dbutils.widgets.text("serie_codigo", "12",           "Código da Série BCB")
dbutils.widgets.text("serie_nome",   "cdi_diario",   "Nome da Série")
dbutils.widgets.text("serie_freq",   "diaria",       "Frequência: diaria|mensal")

In [0]:
import requests
import time
import logging
from datetime import datetime
from pyspark.sql import functions as f
from pyspark.sql.types import StructType, StructField, StringType
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from config import CFG

## Dados BCB - Indicadores de Desempenho 

In [0]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)

In [0]:
# config
SERIE_CODIGO = dbutils.widgets.get("serie_codigo")
SERIE_NOME   = dbutils.widgets.get("serie_nome")
BRONZE_PATH  = f"{CFG.BRONZE_PATH}/bronze_{SERIE_NOME}"
DATA_PROC    = int(datetime.now().strftime("%Y%m%d"))

In [0]:
# HTTP Session com retry
def build_session(retries:int=3, backoff:float=1.5) -> requests.Session:
    session = requests.Session()
    retry = Retry(total=retries, backoff_factor=backoff, status_forcelist=[429, 500, 502, 503, 504])
    session.mount("https://", HTTPAdapter(max_retries=retry))
    return session



In [0]:
SCHEMA_BCB = StructType([
    StructField("data", StringType(), True),
    StructField("valor", StringType(), True),
])


# 1. Extração
hoje    = datetime.today()
inicio  = hoje.replace(year=hoje.year - 10).strftime("%d/%m/%Y")
url     = f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{SERIE_CODIGO}/dados?formato=json&dataInicial={inicio}"

log.info(f"Iniciando ingestão | série={SERIE_CODIGO} | nome={SERIE_NOME} data_processamento={DATA_PROC}")

try:
    session = build_session()
    response = session.get(url, timeout=60)
    response.raise_for_status()
    data = response.json()
except Exception as e:
    log.error(f"Falha na requisição BCB: {e}")
    raise

log.info(f"Registros recebidos da API: {len(data)}")

# 2. Criação do DF com esquema definido
df = spark.createDataFrame(data, schema=SCHEMA_BCB)

# Metadados de rastreabilidade 
df = (df
      .withColumn("_source_url", f.lit(url))
      .withColumn("_ingest_timestamp", f.current_timestamp())
      .withColumn("data_processamento", f.lit(DATA_PROC))
)


In [0]:
# 3. Escrita na bronze
n_registros = df.count()

NOME_TABELA = f"workspace.case_spark_cvm.bronze_{SERIE_NOME}"

log.info(f"Escrevendo {n_registros} linhas em Bronze | path={BRONZE_PATH}")

(df.coalesce(1)
    .write
    .mode("overwrite")
    .option("replaceWhere", f"data_processamento = {DATA_PROC}")
    .option("mergeSchema",  "true")
    .partitionBy("data_processamento")
    .format("delta")
    .saveAsTable(NOME_TABELA)
 )


# 4. metricas
log.info(f"Ingestão concluida | série={SERIE_NOME} | linhas={n_registros} | partição={DATA_PROC}")

In [0]:
%sql
select 
    *
from workspace.case_spark_cvm.bronze_cdi_diario